# Satisfacción de restricciones

Un **problema de satisfacción de restricciones** (*constraint satisfaction problem*, CSP) tiene **variables**, un **dominio** de valores posibles para cada una, y **restricciones** que limitan qué combinaciones están permitidas. Una solución da un valor a cada variable sin incumplir ninguna restricción. Los horarios, los calendarios de exámenes, las configuraciones de productos y los puzles encajan en este molde.

El método básico es el ***backtracking*** (vuelta atrás): dar valor a una variable cada vez y, en cuanto se incumple una restricción, deshacer la última elección y probar el siguiente valor. Qué variable rellenar a continuación importa mucho. La heurística de **mínimos valores restantes** (*minimum remaining values*, MRV) elige la variable con menos valores todavía permitidos: la más restringida, donde un camino equivocado se descubre antes.

## Colorear un mapa

El ejemplo clásico: colorear los estados de Australia con tres colores, de forma que dos estados vecinos no compartan ninguno. Tasmania (T) es una isla sin vecinos.

In [ ]:
states = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
colours = ["red", "green", "blue"]
borders = [("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"), ("SA", "Q"),
           ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V")]
neighbours = {state: set() for state in states}
for state_a, state_b in borders:
    neighbours[state_a].add(state_b)
    neighbours[state_b].add(state_a)

In [ ]:
def allowed_colours(state, assignment):
    return [colour for colour in colours if all(assignment.get(neighbour) != colour for neighbour in neighbours[state])]


def colour_map(assignment, steps):
    if len(assignment) == len(states):
        return assignment
    unassigned = [state for state in states if state not in assignment]
    state = min(unassigned, key=lambda candidate: len(allowed_colours(candidate, assignment)))
    for colour in allowed_colours(state, assignment):
        steps.append(f"{state}={colour}")
        result = colour_map(assignment | {state: colour}, steps)
        if result:
            return result
    return None


steps = []
solution = colour_map({}, steps)
print(solution)
print(len(steps), "assignments:", " ".join(steps))

{'WA': 'red', 'NT': 'green', 'SA': 'blue', 'Q': 'red', 'NSW': 'green', 'V': 'red', 'T': 'red'}
7 assignments: WA=red NT=green SA=blue Q=red NSW=green V=red T=red


Siete asignaciones para siete estados: con MRV la búsqueda nunca tuvo que retroceder.

## Sudoku

Un Sudoku es un CSP con 81 variables, las casillas, cada una con el dominio del 1 al 9, y un solo tipo de restricción: ningún dígito dos veces en una fila, una columna o un cuadro de 3×3. El resolvedor de abajo rellena las casillas vacías mediante *backtracking*, y cuenta cuántos dígitos coloca, incluidos los erróneos. Puede elegir la siguiente casilla de dos formas: la primera casilla vacía en orden de lectura, o la casilla vacía con menos dígitos todavía permitidos (MRV).

In [ ]:
puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]


def allowed_digits(grid, row, column):
    box_row, box_column = 3 * (row // 3), 3 * (column // 3)
    used = set(grid[row]) | {grid[r][column] for r in range(9)} | {
        grid[r][c] for r in range(box_row, box_row + 3) for c in range(box_column, box_column + 3)}
    return [digit for digit in range(1, 10) if digit not in used]

In [ ]:
def solve_sudoku(grid, use_mrv, counter):
    empty_cells = [(row, column) for row in range(9) for column in range(9) if grid[row][column] == 0]
    if not empty_cells:
        return True
    if use_mrv:
        row, column = min(empty_cells, key=lambda cell: len(allowed_digits(grid, *cell)))
    else:
        row, column = empty_cells[0]
    for digit in allowed_digits(grid, row, column):
        grid[row][column] = digit
        counter[0] += 1
        if solve_sudoku(grid, use_mrv, counter):
            return True
    grid[row][column] = 0
    return False

In [ ]:
for use_mrv in [False, True]:
    grid = [row[:] for row in puzzle]
    placements = [0]
    solve_sudoku(grid, use_mrv, placements)
    print(f"{'MRV' if use_mrv else 'Reading order':13}: {placements[0]:5} digits placed")

Reading order:  4208 digits placed
MRV          :    51 digits placed


El puzle tiene 51 casillas vacías, así que 51 colocaciones significa que no hubo ninguna vuelta atrás. La solución:

In [ ]:
for row in grid:
    print(" ".join(str(digit) for digit in row))

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


Los resolvedores reales añaden **propagación de restricciones**: después de cada colocación quitan el dígito de los dominios de las casillas a las que restringe, y abandonan una rama en cuanto alguna casilla se queda sin dígitos (***forward checking***, comprobación hacia delante). Los problemas industriales se resuelven con herramientas dedicadas que combinan todas estas ideas: resolvedores de restricciones y SAT, como OR-Tools de Google, para horarios y rutas, y resolvedores hechos a medida en los gestores de paquetes (pip hace *backtracking* entre versiones; el resolvedor de uv se basa en el algoritmo PubGrub).